# Lab - BUILD: ShopAssist Becomes an Agentic System

Three tasks that turn the loop into one system with case facts, a hook and a handoff. A
loop that runs any tool at any time will, sooner or later, act for an unverified customer
or promise what the policy forbids.

| Task | What you learn |
| --- | --- |
| 1 | The case facts live in your code, and the tools fill them as they run |
| 2 | A hook runs before every tool call and can block it with a structured error |
| 3 | Every call goes through the hook; a refused return goes to a person through escalate |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The system line: the tools check the policy, and escalate hands the case to a person.
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Use the tools for customers, orders and returns; the tools "
    "check the policy and tell you what is missing. If a tool "
    "refuses and the customer still needs help, hand the case "
    "to a person with escalate. Answer in two sentences."
)

CUSTOMERS = {"alex@example.com": "CUS-1001"}
ORDERS = {
    "ORD-87654321": {
        "item": "lamp",
        "status": "delivered",
        "delivered_days_ago": 45,
        "customer_id": "CUS-1001",
    },
}

# Four tools: verify the customer, look up the order, start the return, escalate.
TOOLS = [
    {
        "name": "get_customer",
        "description": "Verify a customer by email. Returns the "
                       "customer ID.",
        "input_schema": {
            "type": "object",
            "properties": {"email": {"type": "string"}},
            "required": ["email"],
        },
    },
    {
        "name": "lookup_order",
        "description": "Look up an order by its ID.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "start_return",
        "description": "Start a return for a delivered order inside "
                       "the 30-day window.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "escalate",
        "description": "Hand the case to a human agent with a short "
                       "reason. Use it when a tool refused and the "
                       "customer still needs help.",
        "input_schema": {
            "type": "object",
            "properties": {"reason": {"type": "string"}},
            "required": ["reason"],
        },
    },
]


# As in the last lab: the policy lives in the tool, and a refusal is a structured error.
def start_return(order_id):
    order = ORDERS[order_id]
    if order["delivered_days_ago"] > 30:
        return {"isError": True, "errorCategory": "business",
                "isRetryable": False,
                "customerMessage": "Past the 30-day return window."}
    return {"return_label": "RET-2048"}

---

## Task 1 - Case facts, filled by the tools

In [ ]:
# ============================================================
# TASK 1 - Case facts, filled by the tools
# ============================================================
#
# WHAT TO DO
#   Keep the case in a small dictionary - the customer ID, the
#   order ID, and whether the case was escalated - and let the
#   tools fill it as they run.
#
# WHY IT MATTERS
#   What the conversation has established - who the customer
#   is, which order this is about - must not depend on what
#   Claude remembers or repeats. It lives in your code, set by
#   the tools that found it. That is what the hook in Task 2
#   reads before it lets a tool run.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Second, the case: a small dictionary".
#
# HOW TO DO IT
#   Three blanks:
#
#       customer_id   get_customer stores what it found
#       order_id      lookup_order stores the order it found
#       True          escalate marks the case as handed over
# ============================================================

case = {"customer_id": None, "order_id": None, "escalated": False}


# TODO: replace each ... below with the value named beside it
def get_customer(email):
    customer_id = CUSTOMERS.get(email)
    if customer_id is None:
        return {"found": False}
    case["customer_id"] = ...   # customer_id - what the tool found
    return {"found": True, "customer_id": customer_id}


def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False, "message": "No order with this ID."}
    case["order_id"] = ...   # order_id - the order that was looked up
    return {"found": True, "order": order}


def escalate(reason):
    case["escalated"] = ...   # True - the case went to a person
    return {"ticket": "ESC-507", "reason": reason}

TOOL_FUNCTIONS = {"get_customer": get_customer,
                  "lookup_order": lookup_order,
                  "start_return": start_return,
                  "escalate": escalate}

print("before:", case)
check("case_facts", case=case, get_customer=get_customer,
      lookup_order=lookup_order, escalate=escalate)

---

## Task 2 - A hook that checks every tool call

In [ ]:
# ============================================================
# TASK 2 - A hook that checks every tool call
# ============================================================
#
# WHAT TO DO
#   Finish before_tool(), the hook. It runs before every tool
#   call and enforces two rules. The gate: no tool except
#   get_customer runs while the customer ID is empty. The order
#   rule: start_return only for the order that was looked up.
#
# WHY IT MATTERS
#   When a rule fails, the hook returns a structured permission
#   or validation error, and the tool never runs. Claude reads
#   the error like any other tool result, and asks the customer
#   for what is missing. When every rule holds, the hook returns
#   None - nothing to object to.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Third, the hook".
#
# HOW TO DO IT
#   Three blanks:
#
#       "get_customer"      the one tool the gate lets through
#       "permission"        the category for "not verified yet"
#       case["order_id"]    the order that was looked up
# ============================================================

# TODO: replace each ... below with the value named beside it
def before_tool(name, args):
    """The hook: runs before every tool call and can block it."""
    if name != ... and case["customer_id"] is None:   # "get_customer"
        return {"isError": True, "errorCategory": ...,   # "permission"
                "customerMessage": "Please give your email first."}
    if name == "start_return" and args["order_id"] != ...:   # case["order_id"]
        return {"isError": True, "errorCategory": "validation",
                "customerMessage": "Look up the order first."}
    return None

print(before_tool("lookup_order", {"order_id": "ORD-87654321"}))   # nobody verified yet

check("hook", before_tool=before_tool, case=case)

---

## Task 3 - Run the loop through the hook

In [ ]:
# ============================================================
# TASK 3 - Run the loop through the hook
# ============================================================
#
# WHAT TO DO
#   In the loop, every tool call goes through the hook first,
#   and the function runs only when the hook returns nothing.
#   Then run two cases: the lamp without an email, and the
#   same customer with their email.
#
# WHY IT MATTERS
#   Without an email, the hook blocks the call and Claude asks
#   for the email - nothing looked up, nothing started. With
#   the email, the customer is verified, the lamp is found,
#   start_return refuses (past the window), and Claude hands the
#   case to a person with escalate. It is tempting to put the
#   gate in the prompt. A prompt is advice; the hook is code,
#   and it cannot be talked around.
#
# WHERE TO SEE IT IN THE LECTURE
#   "every tool call now goes through the hook first".
#
# HOW TO DO IT
#   Three blanks:
#
#       block.name    the tool Claude asked for
#       block.input   the arguments it chose
#       None          the hook found nothing to object to
# ============================================================

# TODO: replace each ... below with the value named beside it
def run(text):
    messages = [{"role": "user", "content": text}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=600,
            output_config={"effort": "low"},
            system=SYSTEM, tools=TOOLS, messages=messages,
        )
        if response.stop_reason != "tool_use":
            break
        messages.append({"role": "assistant",
                         "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = before_tool(..., ...)   # block.name, block.input
                if output is ...:   # None - the hook found nothing to object to
                    output = TOOL_FUNCTIONS[block.name](**block.input)
                print(f"  {block.name}({block.input}) -> {output}")
                results.append({"type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(output),
                                "is_error": bool(output.get("isError"))})
        messages.append({"role": "user", "content": results})
    for block in response.content:
        if block.type == "text":
            print("ShopAssist:", block.text)
    print("case:", case)
    return response, messages   # provided: so the check can read the conversation

CASES = [
    "I want to return the lamp from order ORD-87654321.",
    "Hi, I am alex@example.com. The lamp from order ORD-87654321 "
    "arrived broken and I want to return it.",
]

case.update(customer_id=None, order_id=None, escalated=False)   # provided: a fresh case
try:
    runs = []
    for text in CASES:
        print("Customer:", text)
        response, messages = run(text)
        runs.append((response, messages, dict(case)))
        print()
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except AttributeError as exc:   # provided: a tool call that produced nothing
    if "NoneType" in str(exc):
        raise ValueError("A tool call ended with no output: the hook returned None and "
                         "the function did not run. The blank in `if output is ...:` "
                         "is None.") from None
    raise

check("handoff", runs=runs)

---

## Done

Keep the case facts in your code, let a hook check every tool call before it runs, and
give the loop a handoff tool for the cases the policy cannot settle. Everything in this
system - the loop, the tools, the structured errors - you built in the earlier labs; the
hook and the handoff turn them into one agent you can trust with a customer.